In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as ticker
import textwrap
import matplotlib.patches as mpatches

## ICD-10-CM Chapters

In [ ]:
dataset_paths = {
                "Base":
                    {
                        "Raw":"../data/processed/base_dataset.parquet",
                        "Section Filtered":"../data/processed/section_filtered_dataset.parquet"
                    },
                "Frequent Chapter":{
                    "Without Dropped Section": 
                        {
                            "Base": "../data/processed/frequent_chapter/without_dropped_sections/base_dataset.parquet",
                            "Cleaned Base (ModernBERT)": "../data/processed/frequent_chapter/without_dropped_sections/modernbert_cleaned_base_dataset.parquet",
                            "Cleaned Base (TF-IDF LinearSVC)" :"../data/processed/frequent_chapter/without_dropped_sections/tfidf_cleaned_base_dataset.parquet",
                            "TSO": "../data/processed/frequent_chapter/without_dropped_sections/t_s_dataset.parquet",
                            "Cleaned TSO (ModernBERT)": "../data/processed/frequent_chapter/without_dropped_sections/modernbert_cleaned_t_s_dataset.parquet",
                            "Cleaned TSO (TF-IDF LinearSVC)": "../data/processed/frequent_chapter/without_dropped_sections/tfidf_cleaned_t_s_dataset.parquet"
                        },
                    "With Dropped Section":
                        {
                            "Base": "../data/processed/frequent_chapter/with_dropped_sections/base_dataset.parquet",
                            "Cleaned Base (ModernBERT)": "../data/processed/frequent_chapter/with_dropped_sections/modernbert_cleaned_base_dataset.parquet",
                            "Cleaned Base (TF-IDF LinearSVC)" :"../data/processed/frequent_chapter/with_dropped_sections/tfidf_cleaned_base_dataset.parquet",
                            "TSO": "../data/processed/frequent_chapter/with_dropped_sections/t_s_dataset.parquet",
                            "Cleaned TSO (ModernBERT)": "../data/processed/frequent_chapter/with_dropped_sections/modernbert_cleaned_t_s_dataset.parquet",
                            "Cleaned TSO (TF-IDF LinearSVC)": "../data/processed/frequent_chapter/with_dropped_sections/tfidf_cleaned_t_s_dataset.parquet"
                        },
                     },
                "Top 50 Code":{
                    "Without Dropped Section": 
                        {
                            "Base": "../data/processed/top_50_code/without_dropped_sections/base_dataset.parquet",
                            "Cleaned Base (ModernBERT)": "../data/processed/top_50_code/without_dropped_sections/modernbert_cleaned_base_dataset.parquet",
                            "Cleaned Base (TF-IDF LinearSVC)" :"../data/processed/top_50_code/without_dropped_sections/tfidf_cleaned_base_dataset.parquet",
                            "TSO": "../data/processed/top_50_code/without_dropped_sections/t_s_dataset.parquet",
                            "Cleaned TSO (ModernBERT)": "../data/processed/top_50_code/without_dropped_sections/modernbert_cleaned_t_s_dataset.parquet",
                            "Cleaned TSO (TF-IDF LinearSVC)": "../data/processed/top_50_code/without_dropped_sections/tfidf_cleaned_t_s_dataset.parquet"
                        },
                    "With Dropped Section":
                        {
                            "Base": "../data/processed/top_50_code/with_dropped_sections/base_dataset.parquet",
                            "Cleaned Base (ModernBERT)": "../data/processed/top_50_code/with_dropped_sections/modernbert_cleaned_base_dataset.parquet",
                            "Cleaned Base (TF-IDF LinearSVC)" :"../data/processed/top_50_code/with_dropped_sections/tfidf_cleaned_base_dataset.parquet",
                            "TSO": "../data/processed/top_50_code/with_dropped_sections/t_s_dataset.parquet",
                            "Cleaned TSO (ModernBERT)": "../data/processed/top_50_code/with_dropped_sections/modernbert_cleaned_t_s_dataset.parquet",
                            "Cleaned TSO (TF-IDF LinearSVC)": "../data/processed/top_50_code/with_dropped_sections/tfidf_cleaned_t_s_dataset.parquet"
                        }
                }
}

In [ ]:
ds = pd.read_parquet(dataset_paths["Base"]["Raw"], columns=["chapter"])
df1 = pd.DataFrame({
    "count": ds["chapter"].apply(len),
    "dataset": "Raw"
})

ds = pd.read_parquet(dataset_paths["Base"]["Section Filtered"], columns=["chapter"])
df2 = pd.DataFrame({
    "count": ds["chapter"].apply(len),
    "dataset": "Section Filtered"
})

ds = pd.read_parquet(dataset_paths["Frequent Chapter"]["Without Dropped Section"]["Base"], columns=["chapter"])
df3 = pd.DataFrame({
    "count": ds["chapter"].apply(len),
    "dataset": "Frequent Chapter Base"
})

combined_df = pd.concat([df1, df2, df3], axis=0)

stats = combined_df.groupby("dataset")["count"].agg(['mean', 'median'])

set_stats = {}
for name in stats.index:
    row = stats.loc[name]
    stat = f"{name} (Mean: {row['mean']:.2f} | Median: {row['median']:.0f})"
    set_stats[name] = stat

combined_df["dataset"] = combined_df["dataset"].map(set_stats)

sns.set_theme(style="whitegrid")

counts = ds["chapter"].apply(len)

plt.figure(figsize=(20, 10))

ax = sns.countplot(
    data=combined_df, 
    x="count", 
    hue="dataset", 
    palette="viridis",
    edgecolor="black",
    linewidth=0.5
)

plt.legend(
    title="Datasets",
    frameon=True,
    shadow=True
)
plt.yscale("log")
ax.yaxis.set_major_formatter(ticker.ScalarFormatter())
ax.ticklabel_format(style="plain", axis="y")
plt.title("Distribution of Chapters per Discharge Summary by Dataset")
plt.xlabel("Number of Chapters")
plt.ylabel("Number of Documents")

sns.despine()

ax.xaxis.grid(False) 
plt.tight_layout()

plt.show()

In [ ]:
ds = pd.read_parquet(dataset_paths["Base"]["Raw"], columns=["chapter"])
ds = ds.explode("chapter")["chapter"].value_counts()
df1 = pd.DataFrame({
    "chapter": ds.index,
    "count": ds.values,
    "dataset": "Raw"
})

ds = pd.read_parquet(dataset_paths["Base"]["Section Filtered"], columns=["chapter"])
ds = ds.explode("chapter")["chapter"].value_counts()
df2 = pd.DataFrame({
    "chapter": ds.index,
    "count": ds.values,
    "dataset": "Section Filtered"
})

ds = pd.read_parquet(dataset_paths["Frequent Chapter"]["Without Dropped Section"]["Base"], columns=["chapter"])
ds = ds.explode("chapter")["chapter"].value_counts()
df3 = pd.DataFrame({
    "chapter": ds.index,
    "count": ds.values,
    "dataset": "Frequent Chapter Base"
})

combined_df = pd.concat([df1, df2, df3], axis=0)

order = combined_df.groupby("chapter")["count"].sum().sort_values(ascending=False).index

sns.set_theme(style="whitegrid")
plt.figure(figsize=(22, 11))

ax = sns.barplot(
    data=combined_df,
    y="count",
    x="chapter",
    hue="dataset",
    order=order,
    palette="viridis",
    edgecolor="black",
    linewidth=0.5
)


ax.set_yscale("log")
ax.yaxis.set_major_formatter(ticker.ScalarFormatter())
ax.ticklabel_format(style='plain', axis='y')

plt.title("Frequency of Each Chapter by Dataset")
plt.xlabel("Chapters")
plt.ylabel("Number of Occurrences")
plt.xticks(rotation=90)
plt.legend(
    title="Datasets",
    frameon=True,
    shadow=True
)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
paths = {"Base": dataset_paths["Base"]}
paths.update(dataset_paths["Frequent Chapter"])

stats_list = []
for group_name, datasets in paths.items():
    for ds_name, path in datasets.items():
        df = pd.read_parquet(path, columns=["text"])
        
        word_counts = df["text"].apply(lambda x: len(str(x).split()))
            
        stats_list.append({
            "group": group_name,
            "name": ds_name,
            "min": word_counts.min(),
            "avg": word_counts.mean(),
            "med": word_counts.median(),
            "max": word_counts.max()
        })

df_stats = pd.DataFrame(stats_list)

df_stats["group"] = pd.Categorical(df_stats["group"])

df_stats["key"] = df_stats["group"].astype(str) + " | " + df_stats["name"]

unique_order = df_stats["key"].tolist()

df_melted = df_stats.melt(
    id_vars=["key", "group", "name"], 
    value_vars=["min", "avg", "med", "max"],
    var_name="metric",
    value_name="word_count"
)

metric_order = ["min", "avg", "med", "max"]

palettes = {
    "Base": sns.color_palette("Greens")[2:], 
    "Without Dropped Section": sns.color_palette("Blues")[2:],
    "With Dropped Section": sns.color_palette("Purples")[2:]
}

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(22, 11)) 

sns.barplot(
    data=df_melted,
    x="key",    
    y="word_count",    
    hue="metric",
    hue_order=metric_order,
    order=unique_order,
    ax=ax,
    edgecolor="black",
    linewidth=0.5,
    width=0.95
)


for container_idx, container in enumerate(ax.containers):
    metric_name = metric_order[container_idx]
    
 
    for bar, unique_key in zip(container, unique_order):
        
    
        group_name = df_stats[df_stats["key"] == unique_key]["group"].values[0]
  
        if group_name in palettes:
            color = palettes[group_name][container_idx]
            bar.set_facecolor(color)

    labels = [f"{val:.0f}\n{metric_name}" for val in container.datavalues]
    
    ax.bar_label(
        container, 
        labels=labels,
        padding=5, 
        fontsize=10,
        color='black'
    )


ax.set_yscale("log")
ax.yaxis.set_major_formatter(ticker.ScalarFormatter())
bottom, top = ax.get_ylim()
ax.set_ylim(bottom, top * 5)

clean_labels = [key.split(" | ")[1] for key in unique_order]
wrapped_clean_labels = [textwrap.fill(lbl, 15) for lbl in clean_labels]

ax.set_xticks(range(len(unique_order)))
ax.set_xticklabels(wrapped_clean_labels, fontsize=11)

ax.set_xlabel("Datasets")
ax.set_ylabel("Word Count")
ax.set_title("Word Count Statistics per Dataset")

legend_patches = []
for group_name, colors in palettes.items():
    patch = mpatches.Patch(color=colors[-2], label=group_name)
    legend_patches.append(patch)

ax.legend(
    handles=legend_patches, 
    title="Dataset Groups", 
    loc="upper right",       
    fontsize=12,
    frameon=True,
    shadow=True
)


plt.tight_layout()
plt.show()

## ICD-10-CM Code

In [ ]:
ds = pd.read_parquet(dataset_paths["Base"]["Raw"], columns=["icd_code"])
df1 = pd.DataFrame({
    "count": ds["icd_code"].apply(len),
    "dataset": "Raw"
})

ds = pd.read_parquet(dataset_paths["Base"]["Section Filtered"], columns=["icd_code"])
df2 = pd.DataFrame({
    "count": ds["icd_code"].apply(len),
    "dataset": "Section Filtered"
})

ds = pd.read_parquet(dataset_paths["Top 50 Code"]["Without Dropped Section"]["Base"], columns=["icd_code"])
df3 = pd.DataFrame({
    "count": ds["icd_code"].apply(len),
    "dataset": "Top 50 Code Base"
})

combined_df = pd.concat([df1, df2, df3], axis=0)

stats = combined_df.groupby("dataset")["count"].agg(['mean', 'median'])

set_stats = {}
for name in stats.index:
    row = stats.loc[name]
    stat = f"{name} (Mean: {row['mean']:.2f} | Median: {row['median']:.0f})"
    set_stats[name] = stat

combined_df["dataset"] = combined_df["dataset"].map(set_stats)

sns.set_theme(style="whitegrid")

counts = ds["icd_code"].apply(len)

plt.figure(figsize=(20, 10))

ax = sns.countplot(
    data=combined_df, 
    x="count", 
    hue="dataset", 
    palette="viridis",
    edgecolor="black",
    linewidth=0.5
)

plt.legend(
    title="Datasets",
    frameon=True,
    shadow=True
)

plt.title("Distribution of Codes per Discharge Summary by Dataset")
plt.xlabel("Number of Codes")
plt.ylabel("Number of Documents")

sns.despine()

ax.xaxis.grid(False) 
plt.tight_layout()

plt.show()

In [ ]:
ds = pd.read_parquet(dataset_paths["Base"]["Raw"], columns=["icd_code"])
ds = ds.explode("icd_code")["icd_code"].value_counts()
df1 = pd.DataFrame({
    "icd_code": ds.index,
    "count": ds.values,
    "dataset": "Raw"
})

ds = pd.read_parquet(dataset_paths["Base"]["Section Filtered"], columns=["icd_code"])
ds = ds.explode("icd_code")["icd_code"].value_counts()
df2 = pd.DataFrame({
    "icd_code": ds.index,
    "count": ds.values,
    "dataset": "Section Filtered"
})

ds = pd.read_parquet(dataset_paths["Top 50 Code"]["Without Dropped Section"]["Base"], columns=["icd_code"])
ds = ds.explode("icd_code")["icd_code"].value_counts()
df3 = pd.DataFrame({
    "icd_code": ds.index,
    "count": ds.values,
    "dataset": "Top 50 Code Base"
})

combined_df = pd.concat([df1, df2, df3], axis=0)
combined_df = combined_df[combined_df["icd_code"].isin(ds.index)]

order = combined_df.groupby("icd_code")["count"].sum().sort_values(ascending=False).index

sns.set_theme(style="whitegrid")
plt.figure(figsize=(22, 11))

ax = sns.barplot(
    data=combined_df,
    y="count",
    x="icd_code",
    hue="dataset",
    order=order,
    palette="viridis",
    edgecolor="black",
    linewidth=0.5
)

plt.title("Frequency of Each Code by Dataset")
plt.xlabel("Codes")
plt.ylabel("Number of Occurrences")
plt.xticks(rotation=90)
plt.legend(
    title="Datasets",
    frameon=True,
    shadow=True
)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
paths = {"Base": dataset_paths["Base"]}
paths.update(dataset_paths["Top 50 Code"])

stats_list = []
for group_name, datasets in paths.items():
    for ds_name, path in datasets.items():
        df = pd.read_parquet(path, columns=["text"])
        
        word_counts = df["text"].apply(lambda x: len(str(x).split()))
            
        stats_list.append({
            "group": group_name,
            "name": ds_name,
            "min": word_counts.min(),
            "avg": word_counts.mean(),
            "med": word_counts.median(),
            "max": word_counts.max()
        })

df_stats = pd.DataFrame(stats_list)

df_stats["group"] = pd.Categorical(df_stats["group"])

df_stats["key"] = df_stats["group"].astype(str) + " | " + df_stats["name"]

unique_order = df_stats["key"].tolist()

df_melted = df_stats.melt(
    id_vars=["key", "group", "name"], 
    value_vars=["min", "avg", "med", "max"],
    var_name="metric",
    value_name="word_count"
)

metric_order = ["min", "avg", "med", "max"]

palettes = {
    "Base": sns.color_palette("Greens")[2:], 
    "Without Dropped Section": sns.color_palette("Blues")[2:],
    "With Dropped Section": sns.color_palette("Purples")[2:]
}

sns.set_theme(style="whitegrid")
fig, ax = plt.subplots(figsize=(22, 11)) 

sns.barplot(
    data=df_melted,
    x="key",    
    y="word_count",    
    hue="metric",
    hue_order=metric_order,
    order=unique_order,
    ax=ax,
    edgecolor="black",
    linewidth=0.5,
    width=0.95
)


for container_idx, container in enumerate(ax.containers):
    metric_name = metric_order[container_idx]
    
 
    for bar, unique_key in zip(container, unique_order):
        
    
        group_name = df_stats[df_stats["key"] == unique_key]["group"].values[0]
  
        if group_name in palettes:
            color = palettes[group_name][container_idx]
            bar.set_facecolor(color)

    labels = [f"{val:.0f}\n{metric_name}" for val in container.datavalues]
    
    ax.bar_label(
        container, 
        labels=labels,
        padding=5, 
        fontsize=10,
        color='black'
    )


ax.set_yscale("log")
ax.yaxis.set_major_formatter(ticker.ScalarFormatter())
bottom, top = ax.get_ylim()
ax.set_ylim(bottom, top * 5)

clean_labels = [key.split(" | ")[1] for key in unique_order]
wrapped_clean_labels = [textwrap.fill(lbl, 15) for lbl in clean_labels]

ax.set_xticks(range(len(unique_order)))
ax.set_xticklabels(wrapped_clean_labels, fontsize=11)

ax.set_xlabel("Datasets")
ax.set_ylabel("Word Count")
ax.set_title("Word Count Statistics per Dataset")

legend_patches = []
for group_name, colors in palettes.items():
    patch = mpatches.Patch(color=colors[-2], label=group_name)
    legend_patches.append(patch)

ax.legend(
    handles=legend_patches, 
    title="Dataset Groups", 
    loc="upper right",       
    fontsize=12,
    frameon=True,
    shadow=True
)


plt.tight_layout()
plt.show()